In [1]:
import pandas as pd
import numpy as np
import re
# Set display options to show full content in columns
pd.set_option('display.max_colwidth', None)
import sys
from pathlib import Path

# Get the path to the current script
current_dir = Path.cwd()

# Go one level up
current_dir = current_dir.parent

# Add the 'scripts' directory to sys.path to be able to import data_utils.py
sys.path.append(str(current_dir))

from scripts.data_utils import split_summary_methods, format_mean_std
import ast  
import matplotlib.pyplot as plt
import seaborn as sns
from autorank import autorank, plot_stats, create_report
pd.set_option('display.max_colwidth', None)

In [2]:
MAX_NUMBER_OF_RUNS_STOCASTIC = 3
MAX_SCENARIOS = 2 #20 for Evaluation or 2 Tuning
TYPE_WS = "params_with_p_window_size" #params_with_p_window_size or params_with_window


data = pd.read_excel(current_dir / "datasets" / "summaries" / "summary_results_sota_and_own_method_tuning_pds_and_pv.xlsx")

# Reading PV dataset showing all hyperparameters considered
order_pv = ['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'A9','SA1', 'SA2', 'SA3','DA1', 'DA2', 'DA3','TA1', 'TA2', 'TA3']

order_pds = ['551', '171', '115', '137', '116']


data['method_window_and_param'] = data['method_window_and_param'].str.replace("SWKNN_own", "SWKNN(own)")

data.method_window_and_param.unique()



array(["xStream_2000_{'depth': 25, 'n_chains': 50, 'num_components': 100}",
       "oIF_2000_{'growth_criterion': 'fixed', 'max_leaf_samples': 32, 'n_jobs': -1, 'num_trees': 64}",
       "HStree_2000_{'anomaly_threshold': 0.5, 'max_depth': 15, 'number_of_trees': 15, 'size_limit': 0.1}",
       "HStree_2000_{'anomaly_threshold': 0.5, 'max_depth': 15, 'number_of_trees': 25, 'size_limit': 0.1}",
       "xStream_2000_{'depth': 15, 'n_chains': 50, 'num_components': 50}",
       "oIF_2000_{'growth_criterion': 'adaptive', 'max_leaf_samples': 32, 'n_jobs': -1, 'num_trees': 64}",
       "RobustRandomCutForest_2000_{'num_trees': 4, 'tree_size': 256}",
       "xStream_2000_{'depth': 15, 'n_chains': 50, 'num_components': 100}",
       "KitNet_500_{'hidden_ratio': 0.75, 'learning_rate': 0.1, 'max_size_ae': 10}",
       "SWLOF_500_{'k': 1, 'k_is_max': False, 'metric': 'cityblock', 'simplified': False}",
       "HStree_500_{'anomaly_threshold': 0.5, 'max_depth': 10, 'number_of_trees': 15, 'size_limit

In [3]:
# Find duplicates for the same iteration and the same experiment
duplicates = data[["iteration", "cod_scenario", "method_window_and_param", "count_cleaned_score"]].duplicated(keep=False)
data[duplicates][["iteration", "cod_scenario", "method_window_and_param", "count_cleaned_score"]].sort_values(by=["method_window_and_param"])

,iteration,cod_scenario,method_window_and_param,count_cleaned_score


In [4]:
# Summary of datasets used
pivot = data.pivot_table(values=["count_cleaned_score", "count_anomalies"], index=["cod_scenario"], aggfunc='mean')
pivot['percentage_anomalies'] = (pivot['count_anomalies'] / pivot['count_cleaned_score']) * 100
pivot


,count_anomalies,count_cleaned_score,percentage_anomalies
cod_scenario,,,
116,492.0,10000.0,4.920000
DA3,600.0,4200.0,14.285714


In [5]:
# Adjusting format and creating new columns for analysis in the dataset read

# Extracting new columns for the method, window_size, parameters
data[['method',"window_size", 'parameters']] = data['method_window_and_param'].apply(
    lambda x: pd.Series(split_summary_methods(x))
)

# Creating new column params_with_window
data["params_with_p_window_size"] = "{'p_window_size': " + data["p_window_size"].astype(str)+data["parameters"].str.replace("{", ", ", regex=False)
data["params_with_p_window_size"] = data["params_with_p_window_size"].str.replace("}_{", ",", regex=False)

data["params_with_window"] = "{'window_size': " + data["window_size"].astype(str)+data["parameters"].str.replace("{", ", ", regex=False)
data["params_with_window"] = data["params_with_window"].str.replace("}_{", ",", regex=False)

# Define your list of non-stochastic methods
non_stochastic_methods = ["SWKNN", "SWKNN_own", "SWLOF", "KitNet", "ExactStorm", "mDragStream"]

# Create the 'stochastic' column
data['stochastic'] = data['method'].apply(
    lambda x: 'N' if x in non_stochastic_methods else 'Y')

# Shortening the name of the methods
new_method_name = {"xStream":"XStream", "RSHash":"RSHash", "IForestASD":"IFASD", "RobustRandomCutForest":"RRCF", "KitNet":"KitNet", "ExactStorm":"EStorm","oIF":"OIF", "HStree":"HStree", "OnlineBootKNN":"OBKNN"} 
data["method"] = data['method'].replace(new_method_name)
data["method_with_param"] = data['method']+"_"+data["parameters"]

#Separating mDragstream with Mahalanobis and Euclidean Distance
# Case 1: Euclidean
mask_euc = (data['method'] == 'mDragStream') & \
           (data['params_with_window'].str.contains('euclidean', case=False, na=False))
data.loc[mask_euc, 'method'] = 'mDragStream (Euc)'

# Case 2: Mahalanobis
mask_mah = (data['method'] == 'mDragStream') & \
           (data['params_with_window'].str.contains('mahalanobis', case=False, na=False))
data.loc[mask_mah, 'method'] = 'mDragStream (Mah)'

#Separating OnlineBootKNN with None, Z-normalization or SQRT transformation
# Case 1: None
mask_none = (data['method'] == 'OBKNN') & \
           (data['params_with_window'].str.contains('NONE', case=False, na=False))
data.loc[mask_none, 'method'] = 'OBKNN (TNone)'

# Case 2: ZNormalization
mask_znorm = (data['method'] == 'OBKNN') & \
           (data['params_with_window'].str.contains('ZNORM', case=False, na=False))
data.loc[mask_znorm, 'method'] = 'OBKNN (TZnorm)'

# Case 2: ZNormalization
mask_sqrt = (data['method'] == 'OBKNN') & \
           (data['params_with_window'].str.contains('SQRT', case=False, na=False))
data.loc[mask_sqrt, 'method'] = 'OBKNN (TSqrt)'

# Including the year of publication of the methods
method_and_year = {"XStream":"2018", "RSHash":"2011", "IFASD":"2013", "RRCF":"2016", "KitNet":"2018", "EStorm":"2007","OIF":"2024", "HStree":"2011", "OBKNN (TNone)":"2025", "OBKNN (TZnorm)":"2025", "OBKNN (TSqrt)":"2025", "BKNN":"2025", "SWKNN":"2000","SWLOF":"2000", "SWKNN_own":"2025", "mDragStream (Mah)":"2025", "mDragStream (Euc)":"2025"} 
data["method_and_year"] = data['method'].map(lambda x: f"{x} ({method_and_year.get(x, 'Unknown')})")





In [6]:
# Summary (before data filtering) of Scenarios, number of runs and hyperparameters considered
pivot = data.pivot_table(
    values=[TYPE_WS,"window_size"], 
    index=["cod_scenario","iteration"], 
    columns=['method'], 
    #aggfunc=['count','nunique']
    aggfunc=['nunique']
)
pivot

nunique                      \
                       params_with_p_window_size                       
method                                    EStorm HStree IFASD KitNet   
cod_scenario iteration                                                 
116          0                              15.0   24.0   3.0   24.0   
             1                               NaN   24.0   3.0    NaN   
             2                               NaN   24.0   3.0    NaN   
DA3          0                              15.0   24.0   3.0   24.0   
             1                               NaN   24.0   3.0    NaN   
             2                               NaN   24.0   3.0    NaN   

                                                                              \
                                                                               
method                 OBKNN (TNone) OBKNN (TZnorm)   OIF  RRCF RSHash SWKNN   
cod_scenario iteration                                                         
116          0                  48.0           48.0  24.0  12.0   24.0   9.0   
             1                  48.0           48.0  24.0  12.0   24.0   NaN   
             2                  48.0           48.0  24.0  12.0   24.0   NaN   
DA3          0                  48.0           48.0  24.0  12.0   24.0   9.0   
             1                  48.0           48.0  24.0  12.0   24.0   NaN   
             2                  48.0           48.0  24.0  12.0   24.0   NaN   

                        ...                                                \
                        ...   window_size                                   
method                  ... OBKNN (TNone) OBKNN (TZnorm)  OIF RRCF RSHash   
cod_scenario iteration  ...                                                 
116          0          ...           3.0            3.0  3.0  3.0    3.0   
             1          ...           3.0            3.0  3.0  3.0    3.0   
             2          ...           3.0            3.0  3.0  3.0    3.0   
DA3          0          ...           3.0            3.0  3.0  3.0    3.0   
             1          ...           3.0            3.0  3.0  3.0    3.0   
             2          ...           3.0            3.0  3.0  3.0    3.0   

                                                                                
                                                                                
method                 SWKNN SWLOF XStream mDragStream (Euc) mDragStream (Mah)  
cod_scenario iteration                                                          
116          0           3.0   3.0     3.0               3.0               3.0  
             1           NaN   NaN     3.0               NaN               NaN  
             2           NaN   NaN     3.0               NaN               NaN  
DA3          0           3.0   3.0     3.0               3.0               3.0  
             1           NaN   NaN     3.0               NaN               NaN  
             2           NaN   NaN     3.0               NaN               NaN  

[6 rows x 28 columns]

In [7]:
params_obknn = data[data['method']=="OBKNN"][TYPE_WS].unique()
len(params_obknn)

0

In [8]:
params_obknn

array([], dtype=object)

In [9]:
# Obtaining the best hyperparameters (for all window sizes)
# 1. average results over all iterations and all scenarios
averaged_results = data.groupby(['method', 'parameters', 'cod_scenario'])['raw_pr_auc'].agg(
    mean_auc_pr='mean',
    std_auc_pr='std',
    n_runs='count'
).reset_index()
# 2. average results for each method and hyperparameter (including window size) for all scenarios
summary = averaged_results.groupby(['method','parameters']).agg(
    mean_auc_pr=('mean_auc_pr', 'mean'),
    std_auc_pr=('std_auc_pr', 'mean'),
    n_scenarios=('mean_auc_pr', 'count'),
    #n_experiments=('n_runs', 'sum')
).reset_index()


#summary.sort_values(['mean_auc_pr','std_auc_pr'], ascending=[False, True]).head(10)

# Obtaining the best hyperparameters (for all window sizes)
# 3. Select the highest AUC_PR per method and hyperparameter
best_results = (
    summary
    .sort_values(['mean_auc_pr','std_auc_pr'], ascending=[False, True])
    .groupby('method')
    .head(1)
)
#best_params = best_results['parameters'].unique().tolist()
#best_results


In [10]:
# Obtaining the best window size (for all hyperparameters)
# 1. average results over all iterations and all scenarios
averaged_results = data.groupby(['method', 'p_window_size', 'cod_scenario'])['raw_pr_auc'].agg(
    mean_auc_pr='mean',
    std_auc_pr='std',
    n_runs='count'
).reset_index()
# 2. average results for each method and hyperparameter (including window size) for all scenarios
summary = averaged_results.groupby(['method','p_window_size']).agg(
    mean_auc_pr=('mean_auc_pr', 'mean'),
    std_auc_pr=('std_auc_pr', 'mean'),
    n_scenarios=('mean_auc_pr', 'count'),
    #n_experiments=('n_runs', 'sum')
).reset_index()

#summary.sort_values(['mean_auc_pr','std_auc_pr'], ascending=[False, True]).head(10)

# Obtaining the best window size (for all hyperparameters)
# 3. Select the highest AUC_PR per method and hyperparameter
best_results = (
    summary
    .sort_values(['mean_auc_pr','std_auc_pr'], ascending=[False, True])
    .groupby('method')
    .head(1)
)
#best_ws = best_results[TYPE_WINDOW_SIZE].unique().tolist()
#best_results 

In [11]:
# Obtaining the hyperparameters that are complete for analysis
averaged_results = data.groupby(['method', 'stochastic', TYPE_WS, 'cod_scenario'])['raw_pr_auc'].agg(
    n_runs='count',
).reset_index()

# Condition 1: Stochastic methods must have MAX_NUMBER_OF_RUNS_STOCASTIC
cond_stochastic = (averaged_results['n_runs'] == MAX_NUMBER_OF_RUNS_STOCASTIC) & (averaged_results['stochastic'] == 'Y')

# Condition 2: Non-Stochastic methods must have 1 run
cond_non_stochastic = (averaged_results['n_runs'] == 1) & (averaged_results['stochastic'] == 'N')

# Apply the combined filter using | (OR)
# Let's call this new variable 'filtered_results'
filtered_results = averaged_results[cond_stochastic | cond_non_stochastic]

# Now, 'filtered_results' contains the data you wanted to see
# You can uncomment the next line to inspect it:
# print("--- Filtered Results (Step 1) ---")
# display(filtered_results)


# Now, aggregate the *filtered* results to count complete scenarios
scenario_counts = filtered_results.groupby(['method', TYPE_WS])['cod_scenario'].agg(
    n_scenarios='nunique'
).reset_index()

# Get the final list of complete hyperparameters
scenario_counts = scenario_counts[scenario_counts.n_scenarios==MAX_SCENARIOS]
hyperparameters_complete = scenario_counts[TYPE_WS]


In [12]:
#Applying filters to analyse data

#regex_filter = "|".join(["'transf': 'None'", "shingle_size",'SWKNN'])
regex_filter = "|".join(["mDragStream"])

filtered_data = data[~data['method_window_and_param'].str.contains(regex_filter, regex=True)]

# Filter rows where 'parameters' column contains any of the best_params
#filtered_data = filtered_data[filtered_data['params_with_window'].isin(best_params)]


#filtered_data = filtered_data[filtered_data['method'].str.contains("std_p")]
#filtered_data = filtered_data[filtered_data['method'].str.contains("None|FOD|SOD|DIL|QUANT")]
#filtered_data = filtered_data[filtered_data['method'].str.contains("QUANT")]
#filtered_data = filtered_data[filtered_data.method.str.contains("OnlineBootGP|OnlineBootKNN")]

# Now, sort the data based on the 'scenario' column
filtered_data = filtered_data.sort_values(by='cod_scenario')

#filtered_data = filtered_data[~filtered_data.window_size.isin(["60"])]

#filtered_data = filtered_data[filtered_data.method.isin(["ExactStorm", "OnlineBootKNN"])]
#filtered_data = filtered_data[~filtered_data.cod_scenario.isin(["A10", "A11","A12","TA1", "TA2","TA3"])]
#filtered_data = filtered_data[~filtered_data.cod_scenario.isin(["116"])]
#filtered_data = filtered_data[filtered_data.scenario.isin(["A1", "A2","A3","A4", "A5","A6","A7", "A8","A9"])]


#filtered_data = filtered_data[filtered_data['method_window_and_param'].isin(PARAMS)]
filtered_data = filtered_data[filtered_data[TYPE_WS].isin(hyperparameters_complete)]

#total_scenarios = len(filtered_data.scenario.unique())
# Compute overall mean, std, and count across all datasets
# Exclude the rows where the number of scenarios is not equal to total_scenarios
#summary = summary[summary.n_scenarios==total_scenarios]

#filtered_data.head(10)

In [13]:
# Summary of datasets used (after data filtering)
pivot = filtered_data.pivot_table(values=["count_cleaned_score", "count_anomalies"], index=["cod_scenario"], aggfunc='mean')

pivot['percentage_anomalies'] = (pivot['count_anomalies'] / pivot['count_cleaned_score']) * 100
pivot

,count_anomalies,count_cleaned_score,percentage_anomalies
cod_scenario,,,
116,492.0,10000.0,4.920000
DA3,600.0,4200.0,14.285714


In [14]:
# Summary (after data filtering) of Scenarios, number of runs and hyperparameters considered
pivot = filtered_data.pivot_table(
    values=[TYPE_WS,"window_size"], 
    index=["cod_scenario","iteration"], 
    columns=['method'], 
    #aggfunc=['count','nunique']
    aggfunc=['nunique']
)
pivot

nunique                      \
                       params_with_p_window_size                       
method                                    EStorm HStree IFASD KitNet   
cod_scenario iteration                                                 
116          0                              15.0   24.0   3.0   24.0   
             1                               NaN   24.0   3.0    NaN   
             2                               NaN   24.0   3.0    NaN   
DA3          0                              15.0   24.0   3.0   24.0   
             1                               NaN   24.0   3.0    NaN   
             2                               NaN   24.0   3.0    NaN   

                                                                              \
                                                                               
method                 OBKNN (TNone) OBKNN (TZnorm)   OIF  RRCF RSHash SWKNN   
cod_scenario iteration                                                         
116          0                  48.0           48.0  24.0  12.0   24.0   9.0   
             1                  48.0           48.0  24.0  12.0   24.0   NaN   
             2                  48.0           48.0  24.0  12.0   24.0   NaN   
DA3          0                  48.0           48.0  24.0  12.0   24.0   9.0   
             1                  48.0           48.0  24.0  12.0   24.0   NaN   
             2                  48.0           48.0  24.0  12.0   24.0   NaN   

                        ...                                                  \
                        ... window_size                                       
method                  ...       IFASD KitNet OBKNN (TNone) OBKNN (TZnorm)   
cod_scenario iteration  ...                                                   
116          0          ...         3.0    3.0           3.0            3.0   
             1          ...         3.0    NaN           3.0            3.0   
             2          ...         3.0    NaN           3.0            3.0   
DA3          0          ...         3.0    3.0           3.0            3.0   
             1          ...         3.0    NaN           3.0            3.0   
             2          ...         3.0    NaN           3.0            3.0   

                                                             
                                                             
method                  OIF RRCF RSHash SWKNN SWLOF XStream  
cod_scenario iteration                                       
116          0          3.0  3.0    3.0   3.0   3.0     3.0  
             1          3.0  3.0    3.0   NaN   NaN     3.0  
             2          3.0  3.0    3.0   NaN   NaN     3.0  
DA3          0          3.0  3.0    3.0   3.0   3.0     3.0  
             1          3.0  3.0    3.0   NaN   NaN     3.0  
             2          3.0  3.0    3.0   NaN   NaN     3.0  

[6 rows x 24 columns]

In [15]:
params_obknn = filtered_data[filtered_data['method']=="OBKNN"][TYPE_WS].unique()
len(params_obknn)

0

In [16]:
params_obknn

array([], dtype=object)

In [17]:
# Obtaining the best hyperparameters
# 1. average results over all iterations and all scenarios
averaged_results = filtered_data.groupby(['method', TYPE_WS, 'cod_scenario'])['raw_pr_auc'].agg(
    mean_auc_pr='mean',
    std_auc_pr='std',
    n_runs='count'
).reset_index()

# 2. average results for each method and hyperparameter for all scenarios
summary = averaged_results.groupby(['method', TYPE_WS]).agg(
    mean_auc_pr=('mean_auc_pr', 'mean'),
    beetween_std_auc_pr=('mean_auc_pr', 'std'),
    within_std_auc_pr=('std_auc_pr', 'mean'),
    n_scenarios=('mean_auc_pr', 'count'),
    n_experiments=('n_runs', 'sum')
).reset_index()


#summary.sort_values(['mean_auc_pr','std_auc_pr'], ascending=[False, True]).head(10)

# Obtaining the best hyperparameters 
# 3. Select the highest AUC_PR per method and hyperparameter with lowest standard deviation beetween scenarios
best_results = (
    summary
    .sort_values(['mean_auc_pr','beetween_std_auc_pr'], ascending=[False, True])
    .groupby('method')
    .head(1)
)
best_params = best_results[TYPE_WS].unique().tolist()
best_results

,method,params_with_p_window_size,mean_auc_pr,beetween_std_auc_pr,within_std_auc_pr,n_scenarios,n_experiments
102,OBKNN (TNone),"{'p_window_size': 0.2, 'algorithm': 'brute', 'alpha_ema': 0.01, 'alpha_z_test': 0.05, 'chunk_size': 30, 'dmetric': 'cityblock', 'ensemble_size': 30, 'n_jobs': -1, 'no_bootstrapp': False, 'no_z_score': False, 'transf': 'NONE', 'type_dist': 'largest', 'update_distance_with_abnormal': False, 'update_mode_stats': 'ema'}",0.967986,0.043575,0.003943,2,6
41,IFASD,"{'p_window_size': 0.2, 'initial_window_X': None}",0.959322,0.044329,0.003482,2,6
54,KitNet,"{'p_window_size': 0.05, 'hidden_ratio': 0.9, 'learning_rate': 0.1, 'max_size_ae': 10}",0.943090,0.032492,NaN,2,2
223,SWKNN,"{'p_window_size': 0.02, 'k': 10, 'k_is_max': False, 'max_node_size': 20, 'metric': 'cityblock', 'min_node_size': 5, 'split_sampling': 5}",0.936771,0.089419,NaN,2,2
185,OIF,"{'p_window_size': 0.2, 'growth_criterion': 'fixed', 'max_leaf_samples': 64, 'n_jobs': -1, 'num_trees': 64}",0.648609,0.496942,0.020312,2,6
14,EStorm,"{'p_window_size': 0.2, 'max_radius': 900}",0.557143,0.045457,NaN,2,2
153,OBKNN (TZnorm),"{'p_window_size': 0.2, 'algorithm': 'brute', 'alpha_ema': 0.01, 'alpha_z_test': 0.05, 'chunk_size': 30, 'dmetric': 'cityblock', 'ensemble_size': 30, 'n_jobs': -1, 'no_bootstrapp': False, 'no_z_score': False, 'transf': 'ZNORM', 'type_dist': 'largest', 'update_distance_with_abnormal': True, 'update_mode_stats': 'welford'}",0.548261,0.625520,0.000421,2,6
20,HStree,"{'p_window_size': 0.02, 'anomaly_threshold': 0.5, 'max_depth': 10, 'number_of_trees': 25, 'size_limit': 0.1}",0.480364,0.644116,0.007364,2,6
248,XStream,"{'p_window_size': 0.02, 'depth': 25, 'n_chains': 100, 'num_components': 50}",0.457042,0.083369,0.021800,2,6
234,SWLOF,"{'p_window_size': 0.02, 'k': 10, 'k_is_max': False, 'metric': 'euclidean', 'simplified': False}",0.362281,0.268258,NaN,2,2


## Summary Score of Online Anomaly Detectors with PV Datasets 

In [18]:
# Filtering just the best hyperparameters
filtered_data = filtered_data[filtered_data[TYPE_WS].isin(best_params)]
#filtered_data = filtered_data[filtered_data["method"].isin(["OBKNN (TNone)", "OBKNN (TZnorm)"])]
#filtered_data = filtered_data[filtered_data["method"].isin(["HStree"])]
filtered_data.params_with_p_window_size.unique()

array(["{'p_window_size': 0.02, 'k': 10, 'k_is_max': False, 'max_node_size': 20, 'metric': 'cityblock', 'min_node_size': 5, 'split_sampling': 5}",
       "{'p_window_size': 0.2, 'algorithm': 'brute', 'alpha_ema': 0.01, 'alpha_z_test': 0.05, 'chunk_size': 30, 'dmetric': 'cityblock', 'ensemble_size': 30, 'n_jobs': -1, 'no_bootstrapp': False, 'no_z_score': False, 'transf': 'NONE', 'type_dist': 'largest', 'update_distance_with_abnormal': False, 'update_mode_stats': 'ema'}",
       "{'p_window_size': 0.2, 'algorithm': 'brute', 'alpha_ema': 0.01, 'alpha_z_test': 0.05, 'chunk_size': 30, 'dmetric': 'cityblock', 'ensemble_size': 30, 'n_jobs': -1, 'no_bootstrapp': False, 'no_z_score': False, 'transf': 'ZNORM', 'type_dist': 'largest', 'update_distance_with_abnormal': True, 'update_mode_stats': 'welford'}",
       "{'p_window_size': 0.02, 'depth': 25, 'n_chains': 100, 'num_components': 50}",
       "{'p_window_size': 0.02, 'k': 10, 'k_is_max': False, 'metric': 'euclidean', 'simplified': False}",
 

In [19]:
# Pivot for mean - includes all data
mean_pivot = filtered_data.pivot_table(
    values="raw_pr_auc",
    columns=["cod_scenario", "window_size"],
    index=["method_and_year", TYPE_WS],
    aggfunc="mean"
)

# Pivot for std - includes all data
# Note: This will be NaN for all 'N' rows, which is correct.
std_pivot = filtered_data.pivot_table(
    values="raw_pr_auc",
    columns=["cod_scenario", "window_size"],
    index=["method_and_year", TYPE_WS],
    aggfunc="std"
)


# 2. Stack the DataFrames to align them as Series
mean_s = mean_pivot.stack(level=[0, 1], dropna=False)
std_s = std_pivot.stack(level=[0, 1], dropna=False)

# 3. Apply the formatting function row-by-row
combined_s = pd.DataFrame({'mean': mean_s, 'std': std_s}).apply(
    lambda row: format_mean_std(row['mean'], row['std']), 
    axis=1
)

# 4. Unstack back into a DataFrame
combined = combined_s.unstack(level=[-2, -1])
# Ensure column order is the same as the original pivot
if not combined.empty:
    combined = combined.reindex(columns=mean_pivot.columns)
else:
    # Handle case where combined might be empty
    combined = pd.DataFrame(index=mean_pivot.index, columns=mean_pivot.columns)


# Get the numeric averages first (for sorting and formatting)
avg_mean = mean_pivot.mean(axis=1)
avg_std = std_pivot.mean(axis=1) # This will be NaN for 'N' rows

# 5. Apply the same formatting logic to the Avg column
avg_col_s = pd.DataFrame({'mean': avg_mean, 'std': avg_std}).apply(
    lambda row: format_mean_std(row['mean'], row['std']),
    axis=1
)

combined["Avg"] = avg_col_s

# 6. Sort by the *numeric* average mean (before it was a string)
combined = combined.loc[avg_mean.sort_values(ascending=False).index]

# 7. Display with HTML formatting in Jupyter
combined

C:\Users\nicolas.rojasvarela\AppData\Local\Temp\ipykernel_27732\2208706817.py:20: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  mean_s = mean_pivot.stack(level=[0, 1], dropna=False)
C:\Users\nicolas.rojasvarela\AppData\Local\Temp\ipykernel_27732\2208706817.py:21: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  std_s = std_pivot.stack(level=[0, 1], dropna=False)


cod_scenario                                                                                                                                                                                                                                                                                                                                                         116  \
window_size                                                                                                                                                                                                                                                                                                                                                          200   
method_and_year       params_with_p_window_size                                                                                                                                                                                                                                                                                                                            
OBKNN (TNone) (2025)  {'p_window_size': 0.2, 'algorithm': 'brute', 'alpha_ema': 0.01, 'alpha_z_test': 0.05, 'chunk_size': 30, 'dmetric': 'cityblock', 'ensemble_size': 30, 'n_jobs': -1, 'no_bootstrapp': False, 'no_z_score': False, 'transf': 'NONE', 'type_dist': 'largest', 'update_distance_with_abnormal': False, 'update_mode_stats': 'ema'}                  NaN   
IFASD (2013)          {'p_window_size': 0.2, 'initial_window_X': None}                                                                                                                                                                                                                                                                                               NaN   
KitNet (2018)         {'p_window_size': 0.05, 'hidden_ratio': 0.9, 'learning_rate': 0.1, 'max_size_ae': 10}                                                                                                                                                                                                                                                          NaN   
SWKNN (2000)          {'p_window_size': 0.02, 'k': 10, 'k_is_max': False, 'max_node_size': 20, 'metric': 'cityblock', 'min_node_size': 5, 'split_sampling': 5}                                                                                                                                                                                                1.000 ± NA   
OIF (2024)            {'p_window_size': 0.2, 'growth_criterion': 'fixed', 'max_leaf_samples': 64, 'n_jobs': -1, 'num_trees': 64}                                                                                                                                                                                                                                     NaN   
EStorm (2007)         {'p_window_size': 0.2, 'max_radius': 900}                                                                                                                                                                                                                                                                                                      NaN   
OBKNN (TZnorm) (2025) {'p_window_size': 0.2, 'algorithm': 'brute', 'alpha_ema': 0.01, 'alpha_z_test': 0.05, 'chunk_size': 30, 'dmetric': 'cityblock', 'ensemble_size': 30, 'n_jobs': -1, 'no_bootstrapp': False, 'no_z_score': False, 'transf': 'ZNORM', 'type_dist': 'largest', 'update_distance_with_abnormal': True, 'update_mode_stats': 'welford'}              NaN   
HStree (2011)         {'p_window_size': 0.02, 'anomaly_threshold': 0.5, 'max_depth': 10, 'number_of_trees': 25, 'size_limit': 0.1}                                                                                                                                                                                                                       0.025 ± 2.9e-06

In [22]:
# Pivot for mean - includes all data
mean_pivot = filtered_data.pivot_table(
    values="vus_pr",
    columns=["cod_scenario","window_size"],
    index=["method_and_year", "parameters"],
    aggfunc="mean"
)

# Pivot for std - includes all data
# Note: This will be NaN for all 'N' rows, which is correct.
std_pivot = filtered_data.pivot_table(
    values="vus_pr",
    columns=["cod_scenario", "window_size"],
    index=["method_and_year", "parameters"],
    aggfunc="std"
)


# 2. Stack the DataFrames to align them as Series
mean_s = mean_pivot.stack(level=[0, 1], dropna=False)
std_s = std_pivot.stack(level=[0, 1], dropna=False)

# 3. Apply the formatting function row-by-row
combined_s = pd.DataFrame({'mean': mean_s, 'std': std_s}).apply(
    lambda row: format_mean_std(row['mean'], row['std']), 
    axis=1
)

# 4. Unstack back into a DataFrame
combined = combined_s.unstack(level=[-2, -1])
# Ensure column order is the same as the original pivot
if not combined.empty:
    combined = combined.reindex(columns=mean_pivot.columns)
else:
    # Handle case where combined might be empty
    combined = pd.DataFrame(index=mean_pivot.index, columns=mean_pivot.columns)


# Get the numeric averages first (for sorting and formatting)
avg_mean = mean_pivot.mean(axis=1)
avg_std = std_pivot.mean(axis=1) # This will be NaN for 'N' rows

# 5. Apply the same formatting logic to the Avg column
avg_col_s = pd.DataFrame({'mean': avg_mean, 'std': avg_std}).apply(
    lambda row: format_mean_std(row['mean'], row['std']),
    axis=1
)

combined["Avg"] = avg_col_s

# 6. Sort by the *numeric* average mean (before it was a string)
combined = combined.loc[avg_mean.sort_values(ascending=False).index]

# 7. Display with HTML formatting in Jupyter
combined

C:\Users\nicolas.rojasvarela\AppData\Local\Temp\ipykernel_27732\3280624656.py:20: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  mean_s = mean_pivot.stack(level=[0, 1], dropna=False)
C:\Users\nicolas.rojasvarela\AppData\Local\Temp\ipykernel_27732\3280624656.py:21: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  std_s = std_pivot.stack(level=[0, 1], dropna=False)


cod_scenario                                                                                                                                                                                                                                                                                                                                   116  \
window_size                                                                                                                                                                                                                                                                                                                                    200   
method_and_year       parameters                                                                                                                                                                                                                                                                                                                     
IFASD (2013)          {'initial_window_X': None}                                                                                                                                                                                                                                                                                               NaN   
KitNet (2018)         {'hidden_ratio': 0.9, 'learning_rate': 0.1, 'max_size_ae': 10}                                                                                                                                                                                                                                                           NaN   
SWKNN (2000)          {'k': 10, 'k_is_max': False, 'max_node_size': 20, 'metric': 'cityblock', 'min_node_size': 5, 'split_sampling': 5}                                                                                                                                                                                                 0.999 ± NA   
OBKNN (TNone) (2025)  {'algorithm': 'brute', 'alpha_ema': 0.01, 'alpha_z_test': 0.05, 'chunk_size': 30, 'dmetric': 'cityblock', 'ensemble_size': 30, 'n_jobs': -1, 'no_bootstrapp': False, 'no_z_score': False, 'transf': 'NONE', 'type_dist': 'largest', 'update_distance_with_abnormal': False, 'update_mode_stats': 'ema'}                  NaN   
OIF (2024)            {'growth_criterion': 'fixed', 'max_leaf_samples': 64, 'n_jobs': -1, 'num_trees': 64}                                                                                                                                                                                                                                     NaN   
OBKNN (TZnorm) (2025) {'algorithm': 'brute', 'alpha_ema': 0.01, 'alpha_z_test': 0.05, 'chunk_size': 30, 'dmetric': 'cityblock', 'ensemble_size': 30, 'n_jobs': -1, 'no_bootstrapp': False, 'no_z_score': False, 'transf': 'ZNORM', 'type_dist': 'largest', 'update_distance_with_abnormal': True, 'update_mode_stats': 'welford'}              NaN   
XStream (2018)        {'depth': 25, 'n_chains': 100, 'num_components': 50}                                                                                                                                                                                                                                                         0.568 ± 3.9e-02   
HStree (2011)         {'anomaly_threshold': 0.5, 'max_depth': 10, 'number_of_trees': 25, 'size_limit': 0.1}                                                                                                                                                                                                                        0.035 ± 1.1e-05   
RRCF (2016)           {'num_trees': 4, 'tree_size': 256}                                                                                                                                                                                      

In [23]:
# Generate the report and the plot
# 'alpha' is the significance level
# 'verbose=False' suppresses console output of the report
# 'order='descending' because higher accuracy is better
mean_pivot = filtered_data.pivot_table(
    values="vus_pr",
    index="cod_scenario",
    columns=["method"],
    aggfunc="mean"
)

#result = autorank(mean_pivot, alpha=0.05, verbose=False, order='descending')
#print(result)

#create_report(result)


#plot_stats(result, allow_insignificant=False)
#plt.savefig(current_dir / 'notebooks' / 'img_cdd'/ "cdd_all_methods.pdf", format="pdf", dpi=300, bbox_inches='tight')

plt.show()

In [24]:
# Pivot for mean - includes all data
mean_pivot = filtered_data.pivot_table(
    values="auc_roc",
    columns=["cod_scenario", "window_size"],
    index=["method_and_year", "parameters"],
    aggfunc="mean"
)

# Pivot for std - includes all data
# Note: This will be NaN for all 'N' rows, which is correct.
std_pivot = filtered_data.pivot_table(
    values="auc_roc",
    columns=["cod_scenario", "window_size"],
    index=["method_and_year", "parameters"],
    aggfunc="std"
)


# 2. Stack the DataFrames to align them as Series
mean_s = mean_pivot.stack(level=[0, 1], dropna=False)
std_s = std_pivot.stack(level=[0, 1], dropna=False)

# 3. Apply the formatting function row-by-row
combined_s = pd.DataFrame({'mean': mean_s, 'std': std_s}).apply(
    lambda row: format_mean_std(row['mean'], row['std']), 
    axis=1
)

# 4. Unstack back into a DataFrame
combined = combined_s.unstack(level=[-2, -1])
# Ensure column order is the same as the original pivot
if not combined.empty:
    combined = combined.reindex(columns=mean_pivot.columns)
else:
    # Handle case where combined might be empty
    combined = pd.DataFrame(index=mean_pivot.index, columns=mean_pivot.columns)


# Get the numeric averages first (for sorting and formatting)
avg_mean = mean_pivot.mean(axis=1)
avg_std = std_pivot.mean(axis=1) # This will be NaN for 'N' rows

# 5. Apply the same formatting logic to the Avg column
avg_col_s = pd.DataFrame({'mean': avg_mean, 'std': avg_std}).apply(
    lambda row: format_mean_std(row['mean'], row['std']),
    axis=1
)

combined["Avg"] = avg_col_s

# 6. Sort by the *numeric* average mean (before it was a string)
combined = combined.loc[avg_mean.sort_values(ascending=False).index]

# 7. Display with HTML formatting in Jupyter
combined

C:\Users\nicolas.rojasvarela\AppData\Local\Temp\ipykernel_27732\3052700536.py:20: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  mean_s = mean_pivot.stack(level=[0, 1], dropna=False)
C:\Users\nicolas.rojasvarela\AppData\Local\Temp\ipykernel_27732\3052700536.py:21: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  std_s = std_pivot.stack(level=[0, 1], dropna=False)


cod_scenario                                                                                                                                                                                                                                                                                                                                   116  \
window_size                                                                                                                                                                                                                                                                                                                                    200   
method_and_year       parameters                                                                                                                                                                                                                                                                                                                     
SWKNN (2000)          {'k': 10, 'k_is_max': False, 'max_node_size': 20, 'metric': 'cityblock', 'min_node_size': 5, 'split_sampling': 5}                                                                                                                                                                                                 1.000 ± NA   
KitNet (2018)         {'hidden_ratio': 0.9, 'learning_rate': 0.1, 'max_size_ae': 10}                                                                                                                                                                                                                                                           NaN   
OBKNN (TNone) (2025)  {'algorithm': 'brute', 'alpha_ema': 0.01, 'alpha_z_test': 0.05, 'chunk_size': 30, 'dmetric': 'cityblock', 'ensemble_size': 30, 'n_jobs': -1, 'no_bootstrapp': False, 'no_z_score': False, 'transf': 'NONE', 'type_dist': 'largest', 'update_distance_with_abnormal': False, 'update_mode_stats': 'ema'}                  NaN   
IFASD (2013)          {'initial_window_X': None}                                                                                                                                                                                                                                                                                               NaN   
XStream (2018)        {'depth': 25, 'n_chains': 100, 'num_components': 50}                                                                                                                                                                                                                                                         0.969 ± 1.3e-02   
OIF (2024)            {'growth_criterion': 'fixed', 'max_leaf_samples': 64, 'n_jobs': -1, 'num_trees': 64}                                                                                                                                                                                                                                     NaN   
RRCF (2016)           {'num_trees': 4, 'tree_size': 256}                                                                                                                                                                                                                                                                                       NaN   
SWLOF (2000)          {'k': 10, 'k_is_max': False, 'metric': 'euclidean', 'simplified': False}                                                                                                                                                                                                                                          0.657 ± NA   
OBKNN (TZnorm) (2025) {'algorithm': 'brute', 'alpha_ema': 0.01, 'alpha_z_test': 0.05, 'chunk_size': 30, 'dmetric': 'cityblock', 'ensemble_size': 30, 'n_jobs': -1, 'no_bootstrapp': False, 'no_z_score': False, 'transf': 'ZNORM', 'type_dist'

In [25]:
# Pivot for mean - includes all data
mean_pivot = filtered_data.pivot_table(
    values="raw_pct_detection", # raw_pct_false_positives ,  raw_pct_detection, raw_max_f1
    columns=["cod_scenario", "window_size"],
    index=["method_and_year", "parameters"],
    aggfunc="mean"
)

# Pivot for std - includes all data
# Note: This will be NaN for all 'N' rows, which is correct.
std_pivot = filtered_data.pivot_table(
    values="raw_pct_detection", # raw_pct_false_positives ,  raw_pct_detection, raw_max_f1
    columns=["cod_scenario", "window_size"],
    index=["method_and_year", "parameters"],
    aggfunc="std"
)


# 2. Stack the DataFrames to align them as Series
mean_s = mean_pivot.stack(level=[0, 1], dropna=False)
std_s = std_pivot.stack(level=[0, 1], dropna=False)

# 3. Apply the formatting function row-by-row
combined_s = pd.DataFrame({'mean': mean_s, 'std': std_s}).apply(
    lambda row: format_mean_std(row['mean'], row['std']), 
    axis=1
)

# 4. Unstack back into a DataFrame
combined = combined_s.unstack(level=[-2, -1])
# Ensure column order is the same as the original pivot
if not combined.empty:
    combined = combined.reindex(columns=mean_pivot.columns)
else:
    # Handle case where combined might be empty
    combined = pd.DataFrame(index=mean_pivot.index, columns=mean_pivot.columns)


# Get the numeric averages first (for sorting and formatting)
avg_mean = mean_pivot.mean(axis=1)
avg_std = std_pivot.mean(axis=1) # This will be NaN for 'N' rows

# 5. Apply the same formatting logic to the Avg column
avg_col_s = pd.DataFrame({'mean': avg_mean, 'std': avg_std}).apply(
    lambda row: format_mean_std(row['mean'], row['std']),
    axis=1
)

combined["Avg"] = avg_col_s

# 6. Sort by the *numeric* average mean (before it was a string)
combined = combined.loc[avg_mean.sort_values(ascending=False).index]

# 7. Display with HTML formatting in Jupyter
combined

C:\Users\nicolas.rojasvarela\AppData\Local\Temp\ipykernel_27732\2610500056.py:20: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  mean_s = mean_pivot.stack(level=[0, 1], dropna=False)
C:\Users\nicolas.rojasvarela\AppData\Local\Temp\ipykernel_27732\2610500056.py:21: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  std_s = std_pivot.stack(level=[0, 1], dropna=False)


cod_scenario                                                                                                                                                                                                                                                                                                                                   116  \
window_size                                                                                                                                                                                                                                                                                                                                    200   
method_and_year       parameters                                                                                                                                                                                                                                                                                                                     
EStorm (2007)         {'max_radius': 900}                                                                                                                                                                                                                                                                                                      NaN   
KitNet (2018)         {'hidden_ratio': 0.9, 'learning_rate': 0.1, 'max_size_ae': 10}                                                                                                                                                                                                                                                           NaN   
RSHash (2011)         {'decay': 0.05, 'feature_maxes': [inf, 2000], 'feature_mins': [0], 'num_components': 50, 'num_hash_fns': 1}                                                                                                                                                                                                              NaN   
SWKNN (2000)          {'k': 10, 'k_is_max': False, 'max_node_size': 20, 'metric': 'cityblock', 'min_node_size': 5, 'split_sampling': 5}                                                                                                                                                                                                 1.000 ± NA   
OBKNN (TZnorm) (2025) {'algorithm': 'brute', 'alpha_ema': 0.01, 'alpha_z_test': 0.05, 'chunk_size': 30, 'dmetric': 'cityblock', 'ensemble_size': 30, 'n_jobs': -1, 'no_bootstrapp': False, 'no_z_score': False, 'transf': 'ZNORM', 'type_dist': 'largest', 'update_distance_with_abnormal': True, 'update_mode_stats': 'welford'}              NaN   
OBKNN (TNone) (2025)  {'algorithm': 'brute', 'alpha_ema': 0.01, 'alpha_z_test': 0.05, 'chunk_size': 30, 'dmetric': 'cityblock', 'ensemble_size': 30, 'n_jobs': -1, 'no_bootstrapp': False, 'no_z_score': False, 'transf': 'NONE', 'type_dist': 'largest', 'update_distance_with_abnormal': False, 'update_mode_stats': 'ema'}                  NaN   
IFASD (2013)          {'initial_window_X': None}                                                                                                                                                                                                                                                                                               NaN   
XStream (2018)        {'depth': 25, 'n_chains': 100, 'num_components': 50}                                                                                                                                                                                                                                                         0.937 ± 4.6e-02   
RRCF (2016)           {'num_trees': 4, 'tree_size': 256}                                                                                                                                                                                      

In [26]:
# Pivot for mean - includes all data
mean_pivot = filtered_data.pivot_table(
    values="raw_pct_false_positives", # raw_pct_false_positives ,  raw_pct_detection, raw_max_f1
    columns=["cod_scenario", "window_size"],
    index=["method_and_year", "parameters"],
    aggfunc="mean"
)

# Pivot for std - includes all data
# Note: This will be NaN for all 'N' rows, which is correct.
std_pivot = filtered_data.pivot_table(
    values="raw_pct_false_positives", # raw_pct_false_positives ,  raw_pct_detection, raw_max_f1
    columns=["cod_scenario", "window_size"],
    index=["method_and_year", "parameters"],
    aggfunc="std"
)


# 2. Stack the DataFrames to align them as Series
mean_s = mean_pivot.stack(level=[0, 1], dropna=False)
std_s = std_pivot.stack(level=[0, 1], dropna=False)

# 3. Apply the formatting function row-by-row
combined_s = pd.DataFrame({'mean': mean_s, 'std': std_s}).apply(
    lambda row: format_mean_std(row['mean'], row['std']), 
    axis=1
)

# 4. Unstack back into a DataFrame
combined = combined_s.unstack(level=[-2, -1])
# Ensure column order is the same as the original pivot
if not combined.empty:
    combined = combined.reindex(columns=mean_pivot.columns)
else:
    # Handle case where combined might be empty
    combined = pd.DataFrame(index=mean_pivot.index, columns=mean_pivot.columns)


# Get the numeric averages first (for sorting and formatting)
avg_mean = mean_pivot.mean(axis=1)
avg_std = std_pivot.mean(axis=1) # This will be NaN for 'N' rows

# 5. Apply the same formatting logic to the Avg column
avg_col_s = pd.DataFrame({'mean': avg_mean, 'std': avg_std}).apply(
    lambda row: format_mean_std(row['mean'], row['std']),
    axis=1
)

combined["Avg"] = avg_col_s

# 6. Sort by the *numeric* average mean (before it was a string)
combined = combined.loc[avg_mean.sort_values(ascending=False).index]

# 7. Display with HTML formatting in Jupyter
combined

C:\Users\nicolas.rojasvarela\AppData\Local\Temp\ipykernel_27732\2505021859.py:20: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  mean_s = mean_pivot.stack(level=[0, 1], dropna=False)
C:\Users\nicolas.rojasvarela\AppData\Local\Temp\ipykernel_27732\2505021859.py:21: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  std_s = std_pivot.stack(level=[0, 1], dropna=False)


cod_scenario                                                                                                                                                                                                                                                                                                                                   116  \
window_size                                                                                                                                                                                                                                                                                                                                    200   
method_and_year       parameters                                                                                                                                                                                                                                                                                                                     
EStorm (2007)         {'max_radius': 900}                                                                                                                                                                                                                                                                                                      NaN   
RSHash (2011)         {'decay': 0.05, 'feature_maxes': [inf, 2000], 'feature_mins': [0], 'num_components': 50, 'num_hash_fns': 1}                                                                                                                                                                                                              NaN   
RRCF (2016)           {'num_trees': 4, 'tree_size': 256}                                                                                                                                                                                                                                                                                       NaN   
HStree (2011)         {'anomaly_threshold': 0.5, 'max_depth': 10, 'number_of_trees': 25, 'size_limit': 0.1}                                                                                                                                                                                                                        1.000 ± 0.0e+00   
OBKNN (TZnorm) (2025) {'algorithm': 'brute', 'alpha_ema': 0.01, 'alpha_z_test': 0.05, 'chunk_size': 30, 'dmetric': 'cityblock', 'ensemble_size': 30, 'n_jobs': -1, 'no_bootstrapp': False, 'no_z_score': False, 'transf': 'ZNORM', 'type_dist': 'largest', 'update_distance_with_abnormal': True, 'update_mode_stats': 'welford'}              NaN   
SWLOF (2000)          {'k': 10, 'k_is_max': False, 'metric': 'euclidean', 'simplified': False}                                                                                                                                                                                                                                          0.001 ± NA   
OIF (2024)            {'growth_criterion': 'fixed', 'max_leaf_samples': 64, 'n_jobs': -1, 'num_trees': 64}                                                                                                                                                                                                                                     NaN   
XStream (2018)        {'depth': 25, 'n_chains': 100, 'num_components': 50}                                                                                                                                                                                                                                                         0.022 ± 3.2e-04   
KitNet (2018)         {'hidden_ratio': 0.9, 'learning_rate': 0.1, 'max_size_ae': 10}                                                                                                                                                          

In [27]:
# Pivot for mean - includes all data
mean_pivot = filtered_data.pivot_table(
    values="raw_max_f1", # raw_pct_false_positives ,  raw_pct_detection, raw_max_f1
    columns=["cod_scenario", "window_size"],
    index=["method_and_year", "parameters"],
    aggfunc="mean"
)

# Pivot for std - includes all data
# Note: This will be NaN for all 'N' rows, which is correct.
std_pivot = filtered_data.pivot_table(
    values="raw_max_f1", # raw_pct_false_positives ,  raw_pct_detection, raw_max_f1
    columns=["cod_scenario", "window_size"],
    index=["method_and_year", "parameters"],
    aggfunc="std"
)


# 2. Stack the DataFrames to align them as Series
mean_s = mean_pivot.stack(level=[0, 1], dropna=False)
std_s = std_pivot.stack(level=[0, 1], dropna=False)

# 3. Apply the formatting function row-by-row
combined_s = pd.DataFrame({'mean': mean_s, 'std': std_s}).apply(
    lambda row: format_mean_std(row['mean'], row['std']), 
    axis=1
)

# 4. Unstack back into a DataFrame
combined = combined_s.unstack(level=[-2, -1])
# Ensure column order is the same as the original pivot
if not combined.empty:
    combined = combined.reindex(columns=mean_pivot.columns)
else:
    # Handle case where combined might be empty
    combined = pd.DataFrame(index=mean_pivot.index, columns=mean_pivot.columns)


# Get the numeric averages first (for sorting and formatting)
avg_mean = mean_pivot.mean(axis=1)
avg_std = std_pivot.mean(axis=1) # This will be NaN for 'N' rows

# 5. Apply the same formatting logic to the Avg column
avg_col_s = pd.DataFrame({'mean': avg_mean, 'std': avg_std}).apply(
    lambda row: format_mean_std(row['mean'], row['std']),
    axis=1
)

combined["Avg"] = avg_col_s

# 6. Sort by the *numeric* average mean (before it was a string)
combined = combined.loc[avg_mean.sort_values(ascending=False).index]

# 7. Display with HTML formatting in Jupyter
combined

C:\Users\nicolas.rojasvarela\AppData\Local\Temp\ipykernel_27732\425158449.py:20: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  mean_s = mean_pivot.stack(level=[0, 1], dropna=False)
C:\Users\nicolas.rojasvarela\AppData\Local\Temp\ipykernel_27732\425158449.py:21: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  std_s = std_pivot.stack(level=[0, 1], dropna=False)


cod_scenario                                                                                                                                                                                                                                                                                                                                   116  \
window_size                                                                                                                                                                                                                                                                                                                                    200   
method_and_year       parameters                                                                                                                                                                                                                                                                                                                     
SWKNN (2000)          {'k': 10, 'k_is_max': False, 'max_node_size': 20, 'metric': 'cityblock', 'min_node_size': 5, 'split_sampling': 5}                                                                                                                                                                                                 1.000 ± NA   
OBKNN (TNone) (2025)  {'algorithm': 'brute', 'alpha_ema': 0.01, 'alpha_z_test': 0.05, 'chunk_size': 30, 'dmetric': 'cityblock', 'ensemble_size': 30, 'n_jobs': -1, 'no_bootstrapp': False, 'no_z_score': False, 'transf': 'NONE', 'type_dist': 'largest', 'update_distance_with_abnormal': False, 'update_mode_stats': 'ema'}                  NaN   
IFASD (2013)          {'initial_window_X': None}                                                                                                                                                                                                                                                                                               NaN   
KitNet (2018)         {'hidden_ratio': 0.9, 'learning_rate': 0.1, 'max_size_ae': 10}                                                                                                                                                                                                                                                           NaN   
OIF (2024)            {'growth_criterion': 'fixed', 'max_leaf_samples': 64, 'n_jobs': -1, 'num_trees': 64}                                                                                                                                                                                                                                     NaN   
XStream (2018)        {'depth': 25, 'n_chains': 100, 'num_components': 50}                                                                                                                                                                                                                                                         0.791 ± 2.3e-02   
OBKNN (TZnorm) (2025) {'algorithm': 'brute', 'alpha_ema': 0.01, 'alpha_z_test': 0.05, 'chunk_size': 30, 'dmetric': 'cityblock', 'ensemble_size': 30, 'n_jobs': -1, 'no_bootstrapp': False, 'no_z_score': False, 'transf': 'ZNORM', 'type_dist': 'largest', 'update_distance_with_abnormal': True, 'update_mode_stats': 'welford'}              NaN   
HStree (2011)         {'anomaly_threshold': 0.5, 'max_depth': 10, 'number_of_trees': 25, 'size_limit': 0.1}                                                                                                                                                                                                                        0.093 ± 0.0e+00   
SWLOF (2000)          {'k': 10, 'k_is_max': False, 'metric': 'euclidean', 'simplified': False}                                                                                                                                                

## Summary Training Time of Online Anomaly Detectors with PV Datasets 

In [28]:
# Pivot for mean - includes all data
mean_pivot = filtered_data.pivot_table(
    values="mean_training_time",
    columns=["cod_scenario", "window_size"],
    index=["method_and_year", "parameters"],
    aggfunc="mean"
)

# Pivot for std - includes all data
# Note: This will be NaN for all 'N' rows, which is correct.
std_pivot = filtered_data.pivot_table(
    values="mean_training_time",
    columns=["cod_scenario", "window_size"],
    index=["method_and_year", "parameters"],
    aggfunc="std"
)


# 2. Stack the DataFrames to align them as Series
mean_s = mean_pivot.stack(level=[0, 1], dropna=False)
std_s = std_pivot.stack(level=[0, 1], dropna=False)

# 3. Apply the formatting function row-by-row
combined_s = pd.DataFrame({'mean': mean_s, 'std': std_s}).apply(
    lambda row: format_mean_std(row['mean'], row['std']), 
    axis=1
)

# 4. Unstack back into a DataFrame
combined = combined_s.unstack(level=[-2, -1])
# Ensure column order is the same as the original pivot
if not combined.empty:
    combined = combined.reindex(columns=mean_pivot.columns)
else:
    # Handle case where combined might be empty
    combined = pd.DataFrame(index=mean_pivot.index, columns=mean_pivot.columns)


# Get the numeric averages first (for sorting and formatting)
avg_mean = mean_pivot.mean(axis=1)
avg_std = std_pivot.mean(axis=1) # This will be NaN for 'N' rows

# 5. Apply the same formatting logic to the Avg column
avg_col_s = pd.DataFrame({'mean': avg_mean, 'std': avg_std}).apply(
    lambda row: format_mean_std(row['mean'], row['std']),
    axis=1
)

combined["Avg"] = avg_col_s

# 6. Sort by the *numeric* average mean (before it was a string)
combined = combined.loc[avg_mean.sort_values(ascending=False).index]

# 7. Display with HTML formatting in Jupyter
combined

C:\Users\nicolas.rojasvarela\AppData\Local\Temp\ipykernel_27732\1826925235.py:20: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  mean_s = mean_pivot.stack(level=[0, 1], dropna=False)
C:\Users\nicolas.rojasvarela\AppData\Local\Temp\ipykernel_27732\1826925235.py:21: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  std_s = std_pivot.stack(level=[0, 1], dropna=False)


cod_scenario                                                                                                                                                                                                                                                                                                                                   116  \
window_size                                                                                                                                                                                                                                                                                                                                    200   
method_and_year       parameters                                                                                                                                                                                                                                                                                                                     
OIF (2024)            {'growth_criterion': 'fixed', 'max_leaf_samples': 64, 'n_jobs': -1, 'num_trees': 64}                                                                                                                                                                                                                                     NaN   
XStream (2018)        {'depth': 25, 'n_chains': 100, 'num_components': 50}                                                                                                                                                                                                                                                         0.044 ± 2.7e-05   
KitNet (2018)         {'hidden_ratio': 0.9, 'learning_rate': 0.1, 'max_size_ae': 10}                                                                                                                                                                                                                                                           NaN   
IFASD (2013)          {'initial_window_X': None}                                                                                                                                                                                                                                                                                               NaN   
OBKNN (TZnorm) (2025) {'algorithm': 'brute', 'alpha_ema': 0.01, 'alpha_z_test': 0.05, 'chunk_size': 30, 'dmetric': 'cityblock', 'ensemble_size': 30, 'n_jobs': -1, 'no_bootstrapp': False, 'no_z_score': False, 'transf': 'ZNORM', 'type_dist': 'largest', 'update_distance_with_abnormal': True, 'update_mode_stats': 'welford'}              NaN   
OBKNN (TNone) (2025)  {'algorithm': 'brute', 'alpha_ema': 0.01, 'alpha_z_test': 0.05, 'chunk_size': 30, 'dmetric': 'cityblock', 'ensemble_size': 30, 'n_jobs': -1, 'no_bootstrapp': False, 'no_z_score': False, 'transf': 'NONE', 'type_dist': 'largest', 'update_distance_with_abnormal': False, 'update_mode_stats': 'ema'}                  NaN   
RRCF (2016)           {'num_trees': 4, 'tree_size': 256}                                                                                                                                                                                                                                                                                       NaN   
RSHash (2011)         {'decay': 0.05, 'feature_maxes': [inf, 2000], 'feature_mins': [0], 'num_components': 50, 'num_hash_fns': 1}                                                                                                                                                                                                              NaN   
EStorm (2007)         {'max_radius': 900}                                                                                                                                                                                                     

## Summary Scoring Time of Online Anomaly Detectors with PV Datasets 

In [29]:
# Pivot for mean - includes all data
mean_pivot = filtered_data.pivot_table(
    values="mean_scoring_time",
    columns=["cod_scenario", "window_size"],
    index=["method_and_year", "parameters"],
    aggfunc="mean"
)

# Pivot for std - includes all data
# Note: This will be NaN for all 'N' rows, which is correct.
std_pivot = filtered_data.pivot_table(
    values="mean_scoring_time",
    columns=["cod_scenario", "window_size"],
    index=["method_and_year", "parameters"],
    aggfunc="std"
)


# 2. Stack the DataFrames to align them as Series
mean_s = mean_pivot.stack(level=[0, 1], dropna=False)
std_s = std_pivot.stack(level=[0, 1], dropna=False)

# 3. Apply the formatting function row-by-row
combined_s = pd.DataFrame({'mean': mean_s, 'std': std_s}).apply(
    lambda row: format_mean_std(row['mean'], row['std']), 
    axis=1
)

# 4. Unstack back into a DataFrame
combined = combined_s.unstack(level=[-2, -1])
# Ensure column order is the same as the original pivot
if not combined.empty:
    combined = combined.reindex(columns=mean_pivot.columns)
else:
    # Handle case where combined might be empty
    combined = pd.DataFrame(index=mean_pivot.index, columns=mean_pivot.columns)


# Get the numeric averages first (for sorting and formatting)
avg_mean = mean_pivot.mean(axis=1)
avg_std = std_pivot.mean(axis=1) # This will be NaN for 'N' rows

# 5. Apply the same formatting logic to the Avg column
avg_col_s = pd.DataFrame({'mean': avg_mean, 'std': avg_std}).apply(
    lambda row: format_mean_std(row['mean'], row['std']),
    axis=1
)

combined["Avg"] = avg_col_s

# 6. Sort by the *numeric* average mean (before it was a string)
combined = combined.loc[avg_mean.sort_values(ascending=False).index]

# 7. Display with HTML formatting in Jupyter
combined

C:\Users\nicolas.rojasvarela\AppData\Local\Temp\ipykernel_27732\3238025577.py:20: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  mean_s = mean_pivot.stack(level=[0, 1], dropna=False)
C:\Users\nicolas.rojasvarela\AppData\Local\Temp\ipykernel_27732\3238025577.py:21: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  std_s = std_pivot.stack(level=[0, 1], dropna=False)


cod_scenario                                                                                                                                                                                                                                                                                                                                   116  \
window_size                                                                                                                                                                                                                                                                                                                                    200   
method_and_year       parameters                                                                                                                                                                                                                                                                                                                     
XStream (2018)        {'depth': 25, 'n_chains': 100, 'num_components': 50}                                                                                                                                                                                                                                                         0.043 ± 1.4e-04   
KitNet (2018)         {'hidden_ratio': 0.9, 'learning_rate': 0.1, 'max_size_ae': 10}                                                                                                                                                                                                                                                           NaN   
OBKNN (TZnorm) (2025) {'algorithm': 'brute', 'alpha_ema': 0.01, 'alpha_z_test': 0.05, 'chunk_size': 30, 'dmetric': 'cityblock', 'ensemble_size': 30, 'n_jobs': -1, 'no_bootstrapp': False, 'no_z_score': False, 'transf': 'ZNORM', 'type_dist': 'largest', 'update_distance_with_abnormal': True, 'update_mode_stats': 'welford'}              NaN   
OBKNN (TNone) (2025)  {'algorithm': 'brute', 'alpha_ema': 0.01, 'alpha_z_test': 0.05, 'chunk_size': 30, 'dmetric': 'cityblock', 'ensemble_size': 30, 'n_jobs': -1, 'no_bootstrapp': False, 'no_z_score': False, 'transf': 'NONE', 'type_dist': 'largest', 'update_distance_with_abnormal': False, 'update_mode_stats': 'ema'}                  NaN   
OIF (2024)            {'growth_criterion': 'fixed', 'max_leaf_samples': 64, 'n_jobs': -1, 'num_trees': 64}                                                                                                                                                                                                                                     NaN   
IFASD (2013)          {'initial_window_X': None}                                                                                                                                                                                                                                                                                               NaN   
SWLOF (2000)          {'k': 10, 'k_is_max': False, 'metric': 'euclidean', 'simplified': False}                                                                                                                                                                                                                                          0.000 ± NA   
EStorm (2007)         {'max_radius': 900}                                                                                                                                                                                                                                                                                                      NaN   
SWKNN (2000)          {'k': 10, 'k_is_max': False, 'max_node_size': 20, 'metric': 'cityblock', 'min_node_size': 5, 'split_sampling': 5}                                                                                                       

# Analysis mDragstream

In [30]:
COD_SCENARIO = 'DA3' #Cod of Datasets used for tuning: DA3, 116
filtered_data = data[data["method"].isin(["mDragStream"])]
filtered_data = filtered_data[filtered_data["cod_scenario"].isin([COD_SCENARIO])]

## Euclidean Distance

In [31]:
euclidean_mdragstream = filtered_data[filtered_data['method_window_and_param'].str.contains("'metric': 'euclidean'", na=False)]



In [32]:
# Pivot for mean - includes all data
mean_pivot = euclidean_mdragstream.pivot_table(
    values="raw_pr_auc",
    columns=["cod_scenario", "window_size"],
    index=["method_and_year", "parameters"],
    aggfunc="mean"
)

# Pivot for std - includes all data
# Note: This will be NaN for all 'N' rows, which is correct.
std_pivot = euclidean_mdragstream.pivot_table(
    values="raw_pr_auc",
    columns=["cod_scenario", "window_size"],
    index=["method_and_year", "parameters"],
    aggfunc="std"
)


# 2. Stack the DataFrames to align them as Series
mean_s = mean_pivot.stack(level=[0, 1], dropna=False)
std_s = std_pivot.stack(level=[0, 1], dropna=False)

# 3. Apply the formatting function row-by-row
combined_s = pd.DataFrame({'mean': mean_s, 'std': std_s}).apply(
    lambda row: format_mean_std(row['mean'], row['std']), 
    axis=1
)

# 4. Unstack back into a DataFrame
combined = combined_s.unstack(level=[-2, -1])
# Ensure column order is the same as the original pivot
if not combined.empty:
    combined = combined.reindex(columns=mean_pivot.columns)
else:
    # Handle case where combined might be empty
    combined = pd.DataFrame(index=mean_pivot.index, columns=mean_pivot.columns)


# Get the numeric averages first (for sorting and formatting)
avg_mean = mean_pivot.mean(axis=1)
avg_std = std_pivot.mean(axis=1) # This will be NaN for 'N' rows

# 5. Apply the same formatting logic to the Avg column
avg_col_s = pd.DataFrame({'mean': avg_mean, 'std': avg_std}).apply(
    lambda row: format_mean_std(row['mean'], row['std']),
    axis=1
)

combined["Avg"] = avg_col_s

# 6. Sort by the *numeric* average mean (before it was a string)
combined = combined.loc[avg_mean.sort_values(ascending=False).index]

# 7. Display with HTML formatting in Jupyter
combined

C:\Users\nicolas.rojasvarela\AppData\Local\Temp\ipykernel_27732\965129032.py:20: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  mean_s = mean_pivot.stack(level=[0, 1], dropna=False)
C:\Users\nicolas.rojasvarela\AppData\Local\Temp\ipykernel_27732\965129032.py:21: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  std_s = std_pivot.stack(level=[0, 1], dropna=False)


,cod_scenario,Avg
,window_size,
method_and_year,parameters,


## Mahalanobis Distance

In [33]:
mahalanobis_mdragstream = filtered_data[filtered_data['method_window_and_param'].str.contains("'metric': 'mahalanobis'", na=False)]


In [34]:
# Pivot for mean - includes all data
mean_pivot = mahalanobis_mdragstream.pivot_table(
    values="raw_pr_auc",
    columns=["cod_scenario", "window_size"],
    index=["method_and_year", "parameters"],
    aggfunc="mean"
)

# Pivot for std - includes all data
# Note: This will be NaN for all 'N' rows, which is correct.
std_pivot = mahalanobis_mdragstream.pivot_table(
    values="raw_pr_auc",
    columns=["cod_scenario", "window_size"],
    index=["method_and_year", "parameters"],
    aggfunc="std"
)


# 2. Stack the DataFrames to align them as Series
mean_s = mean_pivot.stack(level=[0, 1], dropna=False)
std_s = std_pivot.stack(level=[0, 1], dropna=False)

# 3. Apply the formatting function row-by-row
combined_s = pd.DataFrame({'mean': mean_s, 'std': std_s}).apply(
    lambda row: format_mean_std(row['mean'], row['std']), 
    axis=1
)

# 4. Unstack back into a DataFrame
combined = combined_s.unstack(level=[-2, -1])
# Ensure column order is the same as the original pivot
if not combined.empty:
    combined = combined.reindex(columns=mean_pivot.columns)
else:
    # Handle case where combined might be empty
    combined = pd.DataFrame(index=mean_pivot.index, columns=mean_pivot.columns)


# Get the numeric averages first (for sorting and formatting)
avg_mean = mean_pivot.mean(axis=1)
avg_std = std_pivot.mean(axis=1) # This will be NaN for 'N' rows

# 5. Apply the same formatting logic to the Avg column
avg_col_s = pd.DataFrame({'mean': avg_mean, 'std': avg_std}).apply(
    lambda row: format_mean_std(row['mean'], row['std']),
    axis=1
)

combined["Avg"] = avg_col_s

# 6. Sort by the *numeric* average mean (before it was a string)
combined = combined.loc[avg_mean.sort_values(ascending=False).index]

# 7. Display with HTML formatting in Jupyter
combined

C:\Users\nicolas.rojasvarela\AppData\Local\Temp\ipykernel_27732\2710109500.py:20: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  mean_s = mean_pivot.stack(level=[0, 1], dropna=False)
C:\Users\nicolas.rojasvarela\AppData\Local\Temp\ipykernel_27732\2710109500.py:21: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  std_s = std_pivot.stack(level=[0, 1], dropna=False)


,cod_scenario,Avg
,window_size,
method_and_year,parameters,
